In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "83fa2e72",
   "metadata": {
    "vscode": {
     "languageId": "plaintext"
    }
   },
   "outputs": [],
   "source": [
    "# %% [markdown]\n",
    "# # OR568 Flight Delay Propagation Data QA + EDA Notebook\n",
    "#\n",
    "# This notebook provides a first-pass quality assessment and exploratory data analysis\n",
    "# for the full-network canonical flight delay propagation datasets.\n",
    "#\n",
    "# It is designed to answer:\n",
    "# - Do the datasets look structurally correct?\n",
    "# - Are timestamps and UTC conversions working?\n",
    "# - Do the aircraft propagation features make sense?\n",
    "# - Does weather coverage look reasonable?\n",
    "# - Are airport-time and route-time aggregates plausible?\n",
    "#\n",
    "# Main datasets:\n",
    "# - flights_canonical_2019.parquet\n",
    "# - aircraft_rotation_2019.parquet\n",
    "# - propagation_chains_2019.parquet\n",
    "# - airport_time_2019.parquet\n",
    "# - route_time_2019.parquet\n",
    "\n",
    "# %%\n",
    "from pathlib import Path\n",
    "import polars as pl\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "# Optional pandas for some convenience views\n",
    "import pandas as pd\n",
    "\n",
    "# Display settings\n",
    "pl.Config.set_tbl_rows(20)\n",
    "pl.Config.set_tbl_cols(20)\n",
    "pl.Config.set_fmt_str_lengths(120)\n",
    "\n",
    "DATA_DIR = Path(\"data/features\")\n",
    "\n",
    "FLIGHTS_PATH = DATA_DIR / \"flights_canonical_2019.parquet\"\n",
    "AIRCRAFT_PATH = DATA_DIR / \"aircraft_rotation_2019.parquet\"\n",
    "CHAINS_PATH = DATA_DIR / \"propagation_chains_2019.parquet\"\n",
    "AIRPORT_TIME_PATH = DATA_DIR / \"airport_time_2019.parquet\"\n",
    "ROUTE_TIME_PATH = DATA_DIR / \"route_time_2019.parquet\"\n",
    "\n",
    "for p in [FLIGHTS_PATH, AIRCRAFT_PATH, CHAINS_PATH, AIRPORT_TIME_PATH, ROUTE_TIME_PATH]:\n",
    "    print(p, \"exists:\", p.exists())\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 1. Load datasets\n",
    "\n",
    "# %%\n",
    "flights = pl.read_parquet(FLIGHTS_PATH)\n",
    "aircraft_rotation = pl.read_parquet(AIRCRAFT_PATH)\n",
    "propagation_chains = pl.read_parquet(CHAINS_PATH)\n",
    "airport_time = pl.read_parquet(AIRPORT_TIME_PATH)\n",
    "route_time = pl.read_parquet(ROUTE_TIME_PATH)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 2. Basic shapes\n",
    "\n",
    "# %%\n",
    "shape_df = pl.DataFrame({\n",
    "    \"dataset\": [\n",
    "        \"flights_canonical_2019\",\n",
    "        \"aircraft_rotation_2019\",\n",
    "        \"propagation_chains_2019\",\n",
    "        \"airport_time_2019\",\n",
    "        \"route_time_2019\",\n",
    "    ],\n",
    "    \"rows\": [\n",
    "        flights.height,\n",
    "        aircraft_rotation.height,\n",
    "        propagation_chains.height,\n",
    "        airport_time.height,\n",
    "        route_time.height,\n",
    "    ],\n",
    "    \"cols\": [\n",
    "        len(flights.columns),\n",
    "        len(aircraft_rotation.columns),\n",
    "        len(propagation_chains.columns),\n",
    "        len(airport_time.columns),\n",
    "        len(route_time.columns),\n",
    "    ],\n",
    "})\n",
    "\n",
    "shape_df\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 3. Schema overview\n",
    "\n",
    "# %%\n",
    "print(\"Flights schema:\")\n",
    "print(flights.schema)\n",
    "\n",
    "# %%\n",
    "print(\"Aircraft rotation schema:\")\n",
    "print(aircraft_rotation.schema)\n",
    "\n",
    "# %%\n",
    "print(\"Propagation chains schema:\")\n",
    "print(propagation_chains.schema)\n",
    "\n",
    "# %%\n",
    "print(\"Airport time schema:\")\n",
    "print(airport_time.schema)\n",
    "\n",
    "# %%\n",
    "print(\"Route time schema:\")\n",
    "print(route_time.schema)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 4. Sample rows\n",
    "\n",
    "# %%\n",
    "flights.head(10)\n",
    "\n",
    "# %%\n",
    "aircraft_rotation.head(10)\n",
    "\n",
    "# %%\n",
    "propagation_chains.head(10)\n",
    "\n",
    "# %%\n",
    "airport_time.head(10)\n",
    "\n",
    "# %%\n",
    "route_time.head(10)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 5. Core integrity checks\n",
    "#\n",
    "# These are the most important first checks:\n",
    "# - row counts\n",
    "# - unique keys\n",
    "# - nulls on critical fields\n",
    "\n",
    "# %%\n",
    "print(\"Flights rows:\", flights.height)\n",
    "print(\"Unique flight_id:\", flights.select(pl.col(\"flight_id\").n_unique()).item())\n",
    "\n",
    "# %%\n",
    "critical_cols = [\n",
    "    \"flight_id\",\n",
    "    \"Origin\",\n",
    "    \"Dest\",\n",
    "    \"Tail_Number\",\n",
    "    \"dep_ts_actual\",\n",
    "    \"arr_ts_actual\",\n",
    "    \"dep_ts_actual_utc\",\n",
    "    \"arr_ts_actual_utc\",\n",
    "]\n",
    "\n",
    "flights.select([\n",
    "    pl.col(c).null_count().alias(c) for c in critical_cols if c in flights.columns\n",
    "])\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 6. Duplicate flight_id check\n",
    "\n",
    "# %%\n",
    "duplicate_flights = (\n",
    "    flights.group_by(\"flight_id\")\n",
    "    .agg(pl.len().alias(\"n\"))\n",
    "    .filter(pl.col(\"n\") > 1)\n",
    "    .sort(\"n\", descending=True)\n",
    ")\n",
    "\n",
    "print(\"Duplicate flight_id rows:\", duplicate_flights.height)\n",
    "duplicate_flights.head(20)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 7. Timestamp sanity checks\n",
    "#\n",
    "# We want to make sure:\n",
    "# - durations are mostly positive\n",
    "# - UTC timestamps exist\n",
    "# - local vs UTC offsets look plausible\n",
    "\n",
    "# %%\n",
    "flights = flights.with_columns([\n",
    "    (pl.col(\"arr_ts_actual_utc\") - pl.col(\"dep_ts_actual_utc\")).dt.total_minutes().alias(\"flight_duration_minutes_utc\"),\n",
    "    (pl.col(\"arr_ts_actual\") - pl.col(\"dep_ts_actual\")).dt.total_minutes().alias(\"flight_duration_minutes_local\"),\n",
    "])\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"flight_duration_minutes_utc\").describe()\n",
    "])\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"flight_duration_minutes_local\").describe()\n",
    "])\n",
    "\n",
    "# %%\n",
    "print(\"Flights with negative UTC duration:\",\n",
    "      flights.filter(pl.col(\"flight_duration_minutes_utc\") < 0).height)\n",
    "\n",
    "print(\"Flights with very long UTC duration (> 1000 mins):\",\n",
    "      flights.filter(pl.col(\"flight_duration_minutes_utc\") > 1000).height)\n",
    "\n",
    "# %%\n",
    "offset_dep = flights.with_columns([\n",
    "    (pl.col(\"dep_ts_actual_utc\") - pl.col(\"dep_ts_actual\")).dt.total_hours().alias(\"dep_utc_offset_hours\")\n",
    "])\n",
    "\n",
    "offset_dep.select(pl.col(\"dep_utc_offset_hours\").describe())\n",
    "\n",
    "# %%\n",
    "offset_arr = flights.with_columns([\n",
    "    (pl.col(\"arr_ts_actual_utc\") - pl.col(\"arr_ts_actual\")).dt.total_hours().alias(\"arr_utc_offset_hours\")\n",
    "])\n",
    "\n",
    "offset_arr.select(pl.col(\"arr_utc_offset_hours\").describe())\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 8. Delay distribution checks\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"DepDelay\").describe(),\n",
    "    pl.col(\"ArrDelay\").describe(),\n",
    "])\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"DepDel15\").mean().alias(\"dep_delay_15_rate\"),\n",
    "    pl.col(\"ArrDel15\").mean().alias(\"arr_delay_15_rate\"),\n",
    "    pl.col(\"Cancelled\").mean().alias(\"cancel_rate\"),\n",
    "    pl.col(\"Diverted\").mean().alias(\"divert_rate\"),\n",
    "])\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 9. Delay histograms\n",
    "#\n",
    "# We clip at a reasonable range so the right tail does not dominate the chart.\n",
    "\n",
    "# %%\n",
    "dep_delay_sample = (\n",
    "    flights.select(\"DepDelay\")\n",
    "    .filter(pl.col(\"DepDelay\").is_not_null())\n",
    "    .filter((pl.col(\"DepDelay\") >= -60) & (pl.col(\"DepDelay\") <= 300))\n",
    "    .sample(min(500_000, flights.height), shuffle=True, seed=42)\n",
    "    .to_pandas()[\"DepDelay\"]\n",
    ")\n",
    "\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.hist(dep_delay_sample, bins=80)\n",
    "plt.xlabel(\"DepDelay (minutes)\")\n",
    "plt.ylabel(\"Count\")\n",
    "plt.title(\"Departure Delay Distribution (clipped)\")\n",
    "plt.show()\n",
    "\n",
    "# %%\n",
    "arr_delay_sample = (\n",
    "    flights.select(\"ArrDelay\")\n",
    "    .filter(pl.col(\"ArrDelay\").is_not_null())\n",
    "    .filter((pl.col(\"ArrDelay\") >= -60) & (pl.col(\"ArrDelay\") <= 300))\n",
    "    .sample(min(500_000, flights.height), shuffle=True, seed=42)\n",
    "    .to_pandas()[\"ArrDelay\"]\n",
    ")\n",
    "\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.hist(arr_delay_sample, bins=80)\n",
    "plt.xlabel(\"ArrDelay (minutes)\")\n",
    "plt.ylabel(\"Count\")\n",
    "plt.title(\"Arrival Delay Distribution (clipped)\")\n",
    "plt.show()\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 10. Propagation sanity checks\n",
    "#\n",
    "# This is one of the most important validation steps.\n",
    "# If the previous arriving flight was late, the next departure should be more likely to be late.\n",
    "\n",
    "# %%\n",
    "prop_check = flights.select([\n",
    "    pl.col(\"DepDel15\").mean().alias(\"overall_dep_del15_rate\")\n",
    "])\n",
    "\n",
    "prop_check\n",
    "\n",
    "# %%\n",
    "prev_late = flights.filter(pl.col(\"prev_arr_late_15\") == 1).select([\n",
    "    pl.len().alias(\"rows\"),\n",
    "    pl.col(\"DepDel15\").mean().alias(\"dep_del15_rate_if_prev_arr_late\"),\n",
    "    pl.col(\"DepDelay\").mean().alias(\"avg_dep_delay_if_prev_arr_late\"),\n",
    "])\n",
    "\n",
    "prev_not_late = flights.filter(pl.col(\"prev_arr_late_15\") == 0).select([\n",
    "    pl.len().alias(\"rows\"),\n",
    "    pl.col(\"DepDel15\").mean().alias(\"dep_del15_rate_if_prev_arr_not_late\"),\n",
    "    pl.col(\"DepDelay\").mean().alias(\"avg_dep_delay_if_prev_arr_not_late\"),\n",
    "])\n",
    "\n",
    "prev_late, prev_not_late\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 11. Turnaround sanity checks\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"turnaround_minutes\").describe(),\n",
    "    pl.col(\"next_turnaround_minutes\").describe(),\n",
    "])\n",
    "\n",
    "# %%\n",
    "print(\"Negative turnaround rows:\",\n",
    "      flights.filter(pl.col(\"turnaround_minutes\") < 0).height)\n",
    "\n",
    "print(\"Very short turnaround (< 20 mins):\",\n",
    "      flights.filter(pl.col(\"turnaround_minutes\") < 20).height)\n",
    "\n",
    "print(\"Very long turnaround (> 1000 mins):\",\n",
    "      flights.filter(pl.col(\"turnaround_minutes\") > 1000).height)\n",
    "\n",
    "# %%\n",
    "turnaround_sample = (\n",
    "    flights.select(\"turnaround_minutes\")\n",
    "    .filter(pl.col(\"turnaround_minutes\").is_not_null())\n",
    "    .filter((pl.col(\"turnaround_minutes\") >= 0) & (pl.col(\"turnaround_minutes\") <= 500))\n",
    "    .sample(min(500_000, flights.height), shuffle=True, seed=42)\n",
    "    .to_pandas()[\"turnaround_minutes\"]\n",
    ")\n",
    "\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.hist(turnaround_sample, bins=80)\n",
    "plt.xlabel(\"Turnaround Minutes\")\n",
    "plt.ylabel(\"Count\")\n",
    "plt.title(\"Turnaround Distribution (clipped)\")\n",
    "plt.show()\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 12. Propagation chain checks\n",
    "\n",
    "# %%\n",
    "propagation_chains.select([\n",
    "    pl.col(\"has_prev_leg\").mean().alias(\"share_has_prev_leg\"),\n",
    "    pl.col(\"has_next_leg\").mean().alias(\"share_has_next_leg\"),\n",
    "    pl.col(\"is_middle_leg\").mean().alias(\"share_middle_leg\"),\n",
    "    pl.col(\"tight_turnaround_lt_60\").mean().alias(\"share_tight_turnaround_lt_60\"),\n",
    "    pl.col(\"tight_turnaround_lt_90\").mean().alias(\"share_tight_turnaround_lt_90\"),\n",
    "])\n",
    "\n",
    "# %%\n",
    "propagation_chains.head(20)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 13. Weather coverage checks\n",
    "\n",
    "# %%\n",
    "weather_cols = [\n",
    "    \"dep_temp_c\",\n",
    "    \"dep_wind_speed_m_s\",\n",
    "    \"dep_wind_dir_deg\",\n",
    "    \"dep_ceiling_height_m\",\n",
    "    \"arr_temp_c\",\n",
    "    \"arr_wind_speed_m_s\",\n",
    "    \"arr_wind_dir_deg\",\n",
    "    \"arr_ceiling_height_m\",\n",
    "]\n",
    "\n",
    "flights.select([\n",
    "    pl.col(c).null_count().alias(c) for c in weather_cols if c in flights.columns\n",
    "])\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"dep_temp_c\").describe(),\n",
    "    pl.col(\"arr_temp_c\").describe(),\n",
    "])\n",
    "\n",
    "# %%\n",
    "flights.select([\n",
    "    pl.col(\"dep_wind_speed_m_s\").describe(),\n",
    "    pl.col(\"arr_wind_speed_m_s\").describe(),\n",
    "])\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 14. Top airports by volume\n",
    "\n",
    "# %%\n",
    "top_origin = (\n",
    "    flights.group_by(\"Origin\")\n",
    "    .agg(pl.len().alias(\"n_flights\"))\n",
    "    .sort(\"n_flights\", descending=True)\n",
    "    .head(20)\n",
    ")\n",
    "\n",
    "top_dest = (\n",
    "    flights.group_by(\"Dest\")\n",
    "    .agg(pl.len().alias(\"n_flights\"))\n",
    "    .sort(\"n_flights\", descending=True)\n",
    "    .head(20)\n",
    ")\n",
    "\n",
    "top_origin, top_dest\n",
    "\n",
    "# %%\n",
    "top_origin_pd = top_origin.to_pandas()\n",
    "\n",
    "plt.figure(figsize=(12, 5))\n",
    "plt.bar(top_origin_pd[\"Origin\"], top_origin_pd[\"n_flights\"])\n",
    "plt.xticks(rotation=45)\n",
    "plt.xlabel(\"Origin Airport\")\n",
    "plt.ylabel(\"Flights\")\n",
    "plt.title(\"Top Origin Airports by Flight Count\")\n",
    "plt.show()\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 15. Top routes by volume\n",
    "\n",
    "# %%\n",
    "top_routes = (\n",
    "    flights.group_by([\"Origin\", \"Dest\"])\n",
    "    .agg(pl.len().alias(\"n_flights\"))\n",
    "    .with_columns(\n",
    "        pl.concat_str([pl.col(\"Origin\"), pl.lit(\" -> \"), pl.col(\"Dest\")]).alias(\"route\")\n",
    "    )\n",
    "    .sort(\"n_flights\", descending=True)\n",
    "    .head(25)\n",
    ")\n",
    "\n",
    "top_routes\n",
    "\n",
    "# %%\n",
    "top_routes_pd = top_routes.head(15).to_pandas()\n",
    "\n",
    "plt.figure(figsize=(12, 5))\n",
    "plt.bar(top_routes_pd[\"route\"], top_routes_pd[\"n_flights\"])\n",
    "plt.xticks(rotation=75)\n",
    "plt.xlabel(\"Route\")\n",
    "plt.ylabel(\"Flights\")\n",
    "plt.title(\"Top Routes by Flight Count\")\n",
    "plt.show()\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 16. Busiest airports by average delay\n",
    "\n",
    "# %%\n",
    "airport_delay = (\n",
    "    flights.group_by(\"Origin\")\n",
    "    .agg([\n",
    "        pl.len().alias(\"n_flights\"),\n",
    "        pl.col(\"DepDelay\").mean().alias(\"avg_dep_delay\"),\n",
    "        pl.col(\"DepDel15\").mean().alias(\"dep_del15_rate\"),\n",
    "    ])\n",
    "    .filter(pl.col(\"n_flights\") > 10000)\n",
    "    .sort(\"avg_dep_delay\", descending=True)\n",
    "    .head(20)\n",
    ")\n",
    "\n",
    "airport_delay\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 17. Airport-time QA\n",
    "\n",
    "# %%\n",
    "airport_time.select([\n",
    "    pl.col(\"total_flight_count\").describe(),\n",
    "    pl.col(\"dep_delay_mean\").describe(),\n",
    "    pl.col(\"arr_delay_mean\").describe(),\n",
    "])\n",
    "\n",
    "# %%\n",
    "airport_time_nulls = airport_time.select([\n",
    "    pl.col(c).null_count().alias(c)\n",
    "    for c in [\"airport\", \"time_bucket\", \"dep_flight_count\", \"arr_flight_count\", \"total_flight_count\"]\n",
    "    if c in airport_time.columns\n",
    "])\n",
    "\n",
    "airport_time_nulls\n",
    "\n",
    "# %%\n",
    "airport_time.group_by(\"airport\").agg(\n",
    "    pl.len().alias(\"n_time_buckets\")\n",
    ").sort(\"n_time_buckets\", descending=True).head(20)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 18. Route-time QA\n",
    "\n",
    "# %%\n",
    "route_time.select([\n",
    "    pl.col(\"flight_count\").describe(),\n",
    "    pl.col(\"dep_delay_mean\").describe(),\n",
    "    pl.col(\"arr_delay_mean\").describe(),\n",
    "])\n",
    "\n",
    "# %%\n",
    "route_time_nulls = route_time.select([\n",
    "    pl.col(c).null_count().alias(c)\n",
    "    for c in [\"Origin\", \"Dest\", \"route_key\", \"time_bucket\", \"flight_count\"]\n",
    "    if c in route_time.columns\n",
    "])\n",
    "\n",
    "route_time_nulls\n",
    "\n",
    "# %%\n",
    "route_time.group_by(\"route_key\").agg(\n",
    "    pl.len().alias(\"n_time_buckets\")\n",
    ").sort(\"n_time_buckets\", descending=True).head(20)\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 19. Compare dataset row relationships\n",
    "\n",
    "# %%\n",
    "relationship_df = pl.DataFrame({\n",
    "    \"metric\": [\n",
    "        \"flights rows\",\n",
    "        \"aircraft_rotation rows\",\n",
    "        \"propagation_chains rows\",\n",
    "        \"airport_time rows\",\n",
    "        \"route_time rows\",\n",
    "    ],\n",
    "    \"value\": [\n",
    "        flights.height,\n",
    "        aircraft_rotation.height,\n",
    "        propagation_chains.height,\n",
    "        airport_time.height,\n",
    "        route_time.height,\n",
    "    ]\n",
    "})\n",
    "\n",
    "relationship_df\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 20. High-level QA summary table\n",
    "\n",
    "# %%\n",
    "summary = {\n",
    "    \"flights_rows\": flights.height,\n",
    "    \"unique_flight_id\": flights.select(pl.col(\"flight_id\").n_unique()).item(),\n",
    "    \"null_dep_ts_actual_utc\": flights.select(pl.col(\"dep_ts_actual_utc\").null_count()).item(),\n",
    "    \"null_arr_ts_actual_utc\": flights.select(pl.col(\"arr_ts_actual_utc\").null_count()).item(),\n",
    "    \"negative_utc_durations\": flights.filter(pl.col(\"flight_duration_minutes_utc\") < 0).height,\n",
    "    \"avg_dep_delay\": flights.select(pl.col(\"DepDelay\").mean()).item(),\n",
    "    \"avg_arr_delay\": flights.select(pl.col(\"ArrDelay\").mean()).item(),\n",
    "    \"dep_del15_rate\": flights.select(pl.col(\"DepDel15\").mean()).item(),\n",
    "    \"arr_del15_rate\": flights.select(pl.col(\"ArrDel15\").mean()).item(),\n",
    "    \"share_has_prev_leg\": propagation_chains.select(pl.col(\"has_prev_leg\").mean()).item(),\n",
    "    \"share_has_next_leg\": propagation_chains.select(pl.col(\"has_next_leg\").mean()).item(),\n",
    "    \"share_middle_leg\": propagation_chains.select(pl.col(\"is_middle_leg\").mean()).item(),\n",
    "}\n",
    "\n",
    "summary_df = pl.DataFrame({\n",
    "    \"metric\": list(summary.keys()),\n",
    "    \"value\": list(summary.values()),\n",
    "})\n",
    "\n",
    "summary_df\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 21. Suggested interpretation guide\n",
    "#\n",
    "# Things that usually indicate the data looks healthy:\n",
    "#\n",
    "# 1. `flight_id` is nearly or fully unique\n",
    "# 2. UTC timestamp nulls are very low\n",
    "# 3. Negative UTC durations are rare\n",
    "# 4. Flights with `prev_arr_late_15 == 1` have meaningfully higher departure delay rates\n",
    "# 5. Top airports/routes look plausible\n",
    "# 6. Weather coverage is substantial\n",
    "# 7. Airport-time and route-time tables have sensible count distributions\n",
    "#\n",
    "# If those checks pass, the canonical dataset is likely in very good shape for:\n",
    "# - regression\n",
    "# - Bayesian models\n",
    "# - LSTMs\n",
    "# - graph-based models\n",
    "# - downstream market-specific hop analysis\n",
    "\n",
    "# %% [markdown]\n",
    "# ## 22. Optional: save QA summary to CSV\n",
    "\n",
    "# %%\n",
    "OUT_DIR = Path(\"data/qa\")\n",
    "OUT_DIR.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "summary_df.write_csv(OUT_DIR / \"qa_summary_2019.csv\")\n",
    "top_origin.write_csv(OUT_DIR / \"top_origin_airports_2019.csv\")\n",
    "top_dest.write_csv(OUT_DIR / \"top_dest_airports_2019.csv\")\n",
    "top_routes.write_csv(OUT_DIR / \"top_routes_2019.csv\")\n",
    "airport_delay.write_csv(OUT_DIR / \"airport_delay_summary_2019.csv\")\n",
    "\n",
    "print(\"Wrote QA outputs to:\", OUT_DIR.resolve())"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}